# Filamentation SiO2 — faisceau convergent, waist en z = 0

Repère simple : **z = 0 est le waist**, la boîte démarre en amont et traverse
le waist. Le solveur est `sim/filament_sim.py` (Éq. (2) de Couairon).

> **Correction par rapport à la première version de ce notebook.** J'avais mis
> `begin = 0`, c'est-à-dire le faisceau lancé *exactement à son waist*. C'est
> physiquement impossible ici : à 12.56 µJ dans 3 µm, l'intensité au waist vaut
> **3.2×10¹⁴ W/cm², soit 6.3× le clampage**. Le milieu s'ionise donc dès le
> premier micron (ρ_e a atteint ρ_max = 2.1×10²², c'est-à-dire *tous* les
> atomes), 82 % de l'énergie est absorbée et il ne reste rien à propager —
> d'où la planche vide. Le run s'est terminé sans erreur, ce qui est
> précisément le piège.
>
> La vraie expérience ne fait pas ça : le faisceau entre dans le verre à
> 394 µm du foyer, où il fait 30 µm de rayon et seulement 3×10¹² W/cm². Il se
> contracte progressivement. **Il faut donc démarrer en amont du waist**, et
> la cellule de contrôle ci-dessous vérifie ce point avant chaque `run()`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e

for p in (Path.cwd().parent / "sim", Path.cwd() / "sim"):
    sys.path.insert(0, str(p))

from filament_sim import run, FIELD_TOGGLES, n_sellmeier
import figures_filament as ff

OUT_ROOT = Path("runs_z0"); OUT_ROOT.mkdir(exist_ok=True)
FIG_DIR = OUT_ROOT / "figures"; FIG_DIR.mkdir(exist_ok=True)
print("toggles disponibles :", FIELD_TOGGLES)

## 1. Paramètres

`z = 0` reste le waist, mais la boîte commence à **−250 µm**. Ce n'est pas un
choix esthétique : l'intensité au waist vaut 6.3× le clampage, et elle décroît
en `I0/(1+(z/z_R)²)` avec `z_R = 39.8 µm`. Il faut donc

| condition | distance minimale au waist |
|---|---|
| `I < I_clamp` (5×10¹³) | 92 µm |
| `I < 10¹³`, MPI négligeable | 221 µm |

`begin = −250 µm` donne `w = 19.1 µm` et `I = 7.8×10¹² W/cm²`, soit 6.4× sous
le clampage — le faisceau se propage linéairement au début et s'auto-focalise
en chemin, comme dans l'expérience.

`w0 = 3 µm` est confirmé par la caustique mesurée (2.84 µm après déconvolution
de la PSF de l'objectif NA 0.28, −5 %). Énergie : 13 µJ incidents → 12.56 µJ
dans le verre après Fresnel, passés **tels quels** à `run()`.

In [ ]:
# ---- laser / matériau ----
WAVELENGTH_M = 1030e-9
W0_M         = 3.0e-6         # waist EN z=0
DELTA_T_S    = 263e-15
ENERGY_IN_GLASS_UJ = 13.0 * (1 - ((1.45-1)/(1.45+1))**2)      # 12.561 uJ

N2          = 2.74e-20        # Milam 1998 @1053nm, le plus proche de 1030
UI_EV       = 9.0
MEFF_REL    = 0.64
TAU_C_S     = 1.7e-15
TAU_R_S     = 330e-15         # piegeage STE (Mouskeftaras 2013)
TAU_STE_S   = 1e-12           # decroissance STE (Sakurai) -- None = STE geles
RHO_MAX_CM3 = 2.1e22
US_EV       = 6.0
F_R, TAU_D_S, TAU_S_S = 0.18, 32e-15, 12e-15
LAMBDA_PROBE_M = 490e-9

# ---- boite : z=0 = waist, on demarre EN AMONT ----
# begin est fixe par l'intensite d'entree, pas par gout : il faut
# |begin| > 92 um pour rester sous I_clamp, > 221 um pour etre a l'aise.
BEGIN_M, END_M = -250e-6, 60e-6

# ---- deux presets de grille ----
FAST = dict(Nz=8000, Nt=2048, Nr=1024, R_factor=20.0,
            save_stride=5, rho_t_stride=16, rho_r_stride=2)
PROD = dict(Nz=16000, Nt=4096, Nr=2048, R_factor=20.0,
            save_stride=10, rho_t_stride=16, rho_r_stride=4)
GRID = FAST                      # <-- bascule ici

n0 = n_sellmeier(WAVELENGTH_M)
k0 = 2*np.pi*n0/WAVELENGTH_M
zR = k0*W0_M**2/2
tp = DELTA_T_S/np.sqrt(2*np.log(2)); tmax = 5*tp
dt = 2*tmax/GRID["Nt"]; dz = (END_M-BEGIN_M)/GRID["Nz"]
R = GRID["R_factor"]*W0_M; dr = R/(GRID["Nr"]-1)
n_saves = GRID["Nz"]//GRID["save_stride"]+1
Nt_sub = (GRID["Nt"]-1)//GRID["rho_t_stride"]+1
Nr_sub = (GRID["Nr"]-2)//GRID["rho_r_stride"]+1

print(f"n0={n0:.4f}  z_R={zR*1e6:.1f} um  (la boite fait {END_M*1e6:.0f} um = {END_M/zR:.1f} z_R)")
print(f"z  : Nz={GRID['Nz']}, dz={dz*1e9:.1f} nm | {n_saves} plans sauves, dz_save={dz*GRID['save_stride']*1e6:.2f} um")
print(f"r  : R_max={R*1e6:.0f} um, dr={dr*1e9:.0f} nm, {W0_M/dr:.0f} pts dans w0")
print(f"t  : Nt={GRID['Nt']}, dt={dt*1e15:.2f} fs, fenetre +/-{tmax*1e15:.0f} fs, f_Nyq/f0={1/(2*dt)/(c_SI/WAVELENGTH_M):.2f}")
print(f"cube (z,r,t) : {n_saves}x{Nr_sub}x{Nt_sub} x3 x4o = {3*n_saves*Nr_sub*Nt_sub*4/1e6:.0f} Mo")
print(f"  dt_sub={dt*GRID['rho_t_stride']*1e15:.1f} fs, dr_sub={dr*GRID['rho_r_stride']*1e9:.0f} nm")

## 2. Où le collapse est-il attendu ?

Le faisceau démarre à son waist, donc `L_DF = k w0²/2 = z_R` directement —
pas de correction de rayon d'entrée cette fois, et pas de focalisation
externe (`f_ext = None`).

In [ ]:
print("=== CONTROLE : intensite au plan d'entree ===")
I_in, w_in, z_safe = ff.check_entrance_intensity(
    ENERGY_IN_GLASS_UJ, W0_M, DELTA_T_S, BEGIN_M, WAVELENGTH_M, n0)

print("\n=== P_cr et longueur de collapse ===")
P_cr = ff.critical_power(N2, WAVELENGTH_M, n0)
P_in = ENERGY_IN_GLASS_UJ*1e-6/(tp*np.sqrt(np.pi/2))
# L_DF doit utiliser le rayon AU PLAN D'ENTREE, pas le waist focal, et la
# focalisation externe est la distance entree->waist.
ratio, L_DF, L_c, L_cf = ff.marburger_collapse(P_in, P_cr, w_in, WAVELENGTH_M,
                                                n0, f_ext=abs(BEGIN_M))
Z_NL_UM = L_cf*1e6 + BEGIN_M*1e6
print(f"P_cr={P_cr*1e-6:.2f} MW  P_in/P_cr={ratio:.1f}")
print(f"L_DF={L_DF*1e6:.0f} um  L_c={L_c*1e6:.0f} um  L_c,f={L_cf*1e6:.0f} um")
print(f"-> foyer non-lineaire attendu vers z = {Z_NL_UM:+.0f} um "
      f"(waist geometrique en z=0)")

## 3. Lancer

`FAST` vise l'itération rapide. Le coût est dominé par la transformée de
Hankel (matrice dense `Nr × Nr` appliquée 4× par pas), donc il croît en
`Nr²·Nt·Nz` — c'est `Nr` qu'il faut baisser en premier si c'est trop lent, pas
`Nz`.

In [ ]:
OUT_DIR = str(OUT_ROOT / f"z0_60um_{GRID['Nz']}x{GRID['Nr']}x{GRID['Nt']}")

res = ff.load_scenario_npz(OUT_DIR)
if res is None:
    print(f"Lancement -> {OUT_DIR}")
    res = run(
        Nz=GRID["Nz"], Nt=GRID["Nt"], Nr=GRID["Nr"], R_factor=GRID["R_factor"],
        begin=BEGIN_M, end=END_M,                 # waist en z=0
        save_stride=GRID["save_stride"], ckpt_every=200, verbose=True,
        wavelength=WAVELENGTH_M, energy_uJ=ENERGY_IN_GLASS_UJ,
        w0=W0_M, delta_t=DELTA_T_S,
        n2=N2, Ui_eV=UI_EV, meff_rel=MEFF_REL,
        tau_c=TAU_C_S, tau_r=TAU_R_S, rho_max=RHO_MAX_CM3,
        Us_eV=US_EV, tau_ste=TAU_STE_S,
        f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S,
        enable_ste=True, lambda_probe=LAMBDA_PROBE_M,
        rho_t_stride=GRID["rho_t_stride"], rho_r_stride=GRID["rho_r_stride"],
        out_dir=OUT_DIR, envelope="gaussian_focused",
    )
print(f"\nU_beam(0) attendu ~ {ENERGY_IN_GLASS_UJ:.2f} uJ")
ff.run_health_check(res, out_dir=OUT_DIR, label="z0_60um", rho_max=RHO_MAX_CM3)

## 4. Diagnostics standard

In [ ]:
VL = [(Z_NL_UM, "foyer non-lineaire (Marburger)", "tab:blue"),
      (0.0, "waist geometrique", "purple")]
ff.plot_fig7_fluence_contours(res, levels=(1.,5.,20.,50.), label="z0, 60 µm",
                              vlines=VL, save=str(FIG_DIR/"fluence.png"))
fig = ff.plot_fig8_peak_intensity({"z0, 60 µm": res})
for zv,l,cc in VL: fig.axes[0].axvline(zv, ls="--", color=cc, lw=1.2, label=l)
fig.axes[0].legend(fontsize=8); fig.savefig(FIG_DIR/"peak_intensity.png", dpi=150)

NC = epsilon_0*m_e*(2*np.pi*c_SI/LAMBDA_PROBE_M)**2/q_e**2*1e-6
ff.plot_free_vs_trapped_vs_z(res, rho_max_cm3=RHO_MAX_CM3, nc_probe_cm3=NC,
                             vlines=VL, save=str(FIG_DIR/"rho.png"))
ff.count_refocusing_cycles(res)

## 5. La planche pompe-sonde — à comparer à l'expérience

Même format que tes données : une colonne par délai, vue de face en haut,
vue de côté en bas, échelle de couleur commune.

**Comment les délais > 1.1 ps sont obtenus.** La fenêtre temporelle du
solveur vaut ±5 t_p = ±1117 fs. Au-delà, `probe_phase_map` n'extrapole pas
naïvement : à t = 5 t_p le champ vaut exp(−25) ≈ 10⁻¹¹ de son maximum, donc
les équations de population se réduisent à deux ODE linéaires
(ρ_e décroît en τ_r, alimente ρ_s qui décroît en τ_STE) dont la solution est
**analytique et exacte**. Vérifié contre une intégration numérique : accord à
10⁻⁸ %.

**Ce que ce modèle ne contient pas.** Aucune physique thermique ni acoustique.
Sur tes données à 3–9 ns on voit clairement une onde de choc qui s'étend :
elle ne sortira jamais de cette simulation. Jusqu'à ~2 ps en revanche, le
déphasage est porté par ρ_e, ρ_s et le Kerr, qui sont tous les trois dans le
modèle.

In [ ]:
DELAYS_FS = [0, 250, 500, 750, 1000, 1500, 2000]

ff.plot_delay_series(
    res, DELAYS_FS,
    lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=4.2, n2=N2,
    tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S,
    z_face_um=None,          # None = plan le plus intense (auto)
    x_half_um=15.0,
    save=str(FIG_DIR/"delay_series.png"),
);

### Décomposition par canal

Si l'amplitude ne colle pas à l'expérience (~0.2 rad au maximum), c'est ici
qu'on voit lequel des trois canaux en est responsable, plutôt que de deviner.

In [ ]:
for chans in (("drude",), ("ste",), ("kerr",), ("drude","ste","kerr")):
    _, _, phi = ff.probe_phase_map(res, 0.0, lambda_probe_m=LAMBDA_PROBE_M,
                                   E_tr_eV=4.2, n2=N2, tau_r_s=TAU_R_S,
                                   tau_ste_s=TAU_STE_S, include=chans, x_half_um=15.0)
    print(f"  a 0 fs, canaux {str(chans):32s} |phi|max = {np.abs(phi).max():6.3f} rad")
print()
for d in DELAYS_FS:
    _, _, phi = ff.probe_phase_map(res, d, lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=4.2,
                                    n2=N2, tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S, x_half_um=15.0)
    print(f"  delai {d:5d} fs : |phi|max = {np.abs(phi).max():6.3f} rad")
print("\nMesure experimentale : ~0.2 rad")

## 6. Itérer

Les trois leviers, par ordre d'effet attendu sur le déphasage :

| levier | comment |
|---|---|
| énergie | `ENERGY_IN_GLASS_UJ` |
| forme du faisceau | `W0_M` — la caustique mesurée donne un profil ~1.6–2.4× moins concentré qu'une gaussienne, donc un `w0` effectif jusqu'à 1.55× plus grand reproduit mieux l'intensité crête réelle |
| canal STE | `TAU_STE_S` (None = STE gelés) et `E_tr_eV` (4.2 dans le dépôt, 5.8 dans le slider — facteur 2.4 sur la contribution STE) |

Et pour isoler un terme physique, les six interrupteurs de l'Éq. (3) sont
disponibles : `run(..., enable_plasma_defocusing=False)` etc.

In [ ]:
# Exemple : meme run avec w0 effectif (profil non gaussien) -- decommenter
# W0_EFF = W0_M*1.55
# OUT_B = str(OUT_ROOT / f"z0_60um_w0eff_{W0_EFF*1e6:.1f}um")
# res_B = ff.load_scenario_npz(OUT_B)
# if res_B is None:
#     res_B = run(Nz=GRID["Nz"], Nt=GRID["Nt"], Nr=GRID["Nr"], R_factor=GRID["R_factor"],
#                 begin=BEGIN_M, end=END_M, save_stride=GRID["save_stride"],
#                 ckpt_every=200, verbose=True, wavelength=WAVELENGTH_M,
#                 energy_uJ=ENERGY_IN_GLASS_UJ, w0=W0_EFF, delta_t=DELTA_T_S,
#                 n2=N2, Ui_eV=UI_EV, meff_rel=MEFF_REL, tau_c=TAU_C_S, tau_r=TAU_R_S,
#                 rho_max=RHO_MAX_CM3, Us_eV=US_EV, tau_ste=TAU_STE_S,
#                 f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S, enable_ste=True,
#                 lambda_probe=LAMBDA_PROBE_M, rho_t_stride=GRID["rho_t_stride"],
#                 rho_r_stride=GRID["rho_r_stride"], out_dir=OUT_B,
#                 envelope="gaussian_focused")
# ff.plot_delay_series(res_B, DELAYS_FS, lambda_probe_m=LAMBDA_PROBE_M, E_tr_eV=4.2,
#                      n2=N2, tau_r_s=TAU_R_S, tau_ste_s=TAU_STE_S,
#                      z_face_um=None, x_half_um=15.0,
#                      save=str(FIG_DIR/"delay_series_w0eff.png"));